In [ ]:
from app.models.statements import Comment, ConversationStarter
from app.i18n import Topic

import asyncio
import random


In [ ]:
starters = [
    ConversationStarter(text="start", topics=("belgian_democracy",)),
    ConversationStarter(text="wat gebeurt er?", topics=("everyday_democracy",))
           ]
            
comments = [
    Comment(text="start yourself", 
                   reply_to=starters[0].ID, topics=starters[0].topics),
    Comment(text="er gebeurt helemaal niets.", 
                   reply_to=starters[1].ID, topics=starters[1].topics),
    Comment(text="un commentaire nouveau par moi", 
                   reply_to=None, topics=("protest",)),
]

comments.append(
    Comment(text="une reponse par toi", 
                   reply_to=comments[-1].ID, topics=comments[-1].topics)
)

comments = {c.ID: c for c in comments}

In [ ]:
comments

---
## figuring out queueing of comments

In [ ]:
strings = [
    "I completely disagree with this article. The author clearly hasn't done their research.",
    "Dit is precies wat ik al jaren zeg, maar niemand luistert.",
    "Franchement, je ne comprends pas pourquoi les gens s'énervent autant à ce sujet.",
    "Great point, but I think you're missing the bigger picture here.",
    "Wie heeft dit geschreven? Dit slaat echt nergens op.",
    "Merci pour cet article, c'est très bien expliqué.",
    "Honestly, this is the best take I've read on the topic so far.",
    "Ik ben het er niet mee eens. De cijfers zeggen iets heel anders.",
    "C'est n'importe quoi, on nous prend vraiment pour des idiots.",
    "Can someone explain why this is even news? Feels like clickbait.",
]

from app.i18n import Topic
import numpy as np

rng = np.random.default_rng()


def sample_topics(min_n=1, max_n=None):
      members = list(Topic)
      if max_n is None: max_n = len(members)
      return tuple(random.sample(members, max(1, int(rng.poisson(lam=2)))))

def sample_new_comment():
    reply_to = (random.choice(list(comments.values())).ID if random.random() > 0.4
                        else None)
    topics = (sample_topics() if not reply_to else comments[reply_to].topics)
    new_comm = Comment(text=random.choice(strings),
                        reply_to=reply_to,
                        topics=topics)
    return new_comm


class SampledConversation:
    def __init__(self, starter=None):
        if starter is None:
            self.start = random.choice(starters)
            self.emitted = 0

        else:
            self.start = starter
            self.emitted = 1
        self.cur = self.start
    
    def __iter__(self):
        return self


    def sample_next(self):
        return random.choice(list(comments.values()))

    def __next__(self):
        if (self.emitted < 4) or (random.random() > 0.5):
            to_return = self.cur
            self.cur = self.sample_next()
            self.emitted += 1
            return to_return 
        else:
            raise StopIteration   


In [ ]:
async def receiver_loop(queue):
    while True:
        await asyncio.sleep(random.uniform(0.3, 4.0))
        if random.random() < 0.5:
            # print("sampling...", flush=True)
            new_comment = sample_new_comment()                   
            await queue.put(new_comment)


async def sender_loop(queue, interval=2, min_len_without_interception=3):
    loop = asyncio.get_running_loop()
    next_at = loop.time()
    without_interception = min_len_without_interception
    convo = SampledConversation()
    while True:
        next_at += interval
        await asyncio.sleep(max(0, next_at - loop.time()))

        if without_interception < 1 and not queue.empty():
            cur = queue.get_nowait()
            print("-"*10, f"\n{loop.time():8.2f}: NEW COMMENT\t\n{(cur.ID, cur.reply_to)}\n", 
                  "-"*10, "\n")
            without_interception = min_len_without_interception
            comments[cur.ID] = cur
            # here, this comment starts its own conversation together with its parent
            # (if there is one)
            convo = SampledConversation(cur)
        else:
            try:
                cur = next(convo)
            except StopIteration:
                convo = SampledConversation()
                cur = next(convo)
            if isinstance(cur, ConversationStarter):
                print(f"{loop.time():8.2f}: STARTER {cur.ID}\n")
            else:
                print(f"{loop.time():8.2f}: {(cur.ID, cur.reply_to)}\n")
            without_interception -= 1
            # here, the conversation simply either continues 
            # or ends and a new one is started


async def main():
    queue = asyncio.Queue()
    task = asyncio.create_task(receiver_loop(queue))
    try:
        await sender_loop(queue)
    # except asyncio.CancelledError:
    #     print("--- done ---")
    finally:
        task.cancel()
        await asyncio.gather(task, return_exceptions=True)
        print("done")

await main()   # asyncio.run(main()) in a script


### TODO

 - finish conersation starter & comment and new comment flow
 - add a connection manager back in  
   -> does: (1) broadcast to all open connections, (2) listen to sumission on all connections (necessary?)

---

In [ ]:
from app.models.statements import ConversationStarter
from app.models.render import ConversationStarterRender

In [ ]:
ConversationStarter(text="hoe gaat het?", topics=("belgian_democracy",), language="nl")

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
r = np.random.randint(3, 200, size=500)

plt.hist(r)

In [ ]:
lo, hi = 1, 4

plt.hist((r+1)/200*hi)

In [ ]:
normed = (r - r.min())/(r.max() - r.min())
(normed * 3)+1

In [ ]:
from app.models import Comment, CommentCreateMessage, Flag, FlagCreateMessage, incoming_adapter, render
from app.models.render import CommentRender

In [ ]:
from app.models import Comment, ConversationStarter, ConversationStarterRender
from app.models import render

In [ ]:
c = Comment(text="start yourself", 
                   reply_to=None, topics=("belgian_democracy",))

In [ ]:
crm = render(c, slot="top", new=False)

In [ ]:
crm.model_dump(mode="json")

In [ ]:
cs = ConversationStarter(text="start", topics=("belgian_democracy",))


ConversationStarterRender.from_ConversationStarter(cs).model_dump(mode="json")

---
## data

In [3]:
from app.data import load_conversation_starters
import pandas as pd

# starters = load_conversation_starters(
#     path="./data/conversation_starters/conversation_starters_20260821_translated.csv")

KeyError: 'text'